# Flat File Ingestion Demo

Demonstrates ingesting data into Snowflake from **Azure Blob Storage** using:

1. **Infer Schema** — auto-create a table from CSV metadata
2. **Schema Evolution** — table auto-adapts when new columns appear in files
3. **Snowpipe** — event-driven auto-ingest as files land in Azure
4. **Validation Mode** — inspect bad records before loading
5. **XML Loading** — load and flatten XML into structured tables

**Snowflake supports the following file formats:**

- **Delimited files** — CSV, TSV, and any custom delimiter
- **Semi-structured**
  - JSON
  - XML
  - Parquet
  - Avro
  - ORC

In [ ]:
%%sql -r dataframe_2
-- Set Context
USE ROLE ACCOUNTADMIN;

CREATE DATABASE IF NOT EXISTS INGEST_DEMO;
CREATE SCHEMA IF NOT EXISTS INGEST_DEMO.CLOUDBUCKET;

USE DATABASE INGEST_DEMO;
USE SCHEMA CLOUDBUCKET;

-- Create Stage pointing to Azure Blob container
CREATE OR REPLACE STAGE MY_STAGE
  STORAGE_INTEGRATION = azure_integration
  URL = 'azure://timjones.blob.core.windows.net/data';

-- CSV file format with PARSE_HEADER so column names come from row 1
-- ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE is required for schema evolution
CREATE OR REPLACE FILE FORMAT my_csv_file_format
    TYPE = 'CSV'
    PARSE_HEADER = TRUE
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE;

-- Confirm files are visible in the stage
LIST @MY_STAGE/ingest_demo/;

## Infer Schema & Schema Evolution

[Docs: Schema Evolution](https://docs.snowflake.com/en/user-guide/data-load-schema-evolution)

`INFER_SCHEMA` reads the file headers and returns the detected column names and types — no need to manually define a `CREATE TABLE` DDL.

Requirements for schema evolution:
1. `ENABLE_SCHEMA_EVOLUTION = TRUE` on the table
2. `MATCH_BY_COLUMN_NAME` on the `COPY INTO`

In [ ]:
%%sql -r dataframe_4
-- Create table automatically from the inferred schema
-- ENABLE_SCHEMA_EVOLUTION = TRUE means the table will auto-ALTER when new columns appear
CREATE OR REPLACE TABLE pharmacy_claims
  ENABLE_SCHEMA_EVOLUTION = TRUE
  USING TEMPLATE (
    SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
      FROM TABLE(
        INFER_SCHEMA(
          LOCATION => '@MY_STAGE/ingest_demo/csv_example/'
        , FILE_FORMAT => 'my_csv_file_format'
        , FILES => ('pharmacy_claims.csv')
        )
      ));

-- Confirm table structure (matches inferred columns above)
SELECT * FROM pharmacy_claims;

In [ ]:
%%sql -r dataframe_5
-- Load the CSV into the table
COPY INTO pharmacy_claims
FROM @my_stage/ingest_demo/csv_example/
  FILES = ('pharmacy_claims.csv')
  FILE_FORMAT = (FORMAT_NAME = 'my_csv_file_format')
  MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

-- Verify the load
SELECT * FROM pharmacy_claims;

## Validation Mode

Before loading a file with bad records, use `VALIDATION_MODE = 'RETURN_ERRORS'` to preview errors without writing any data.

## Error Handling
`ON_ERROR` options:
| Option | Behavior |
|---|---|
| `ABORT_STATEMENT` | Stop the entire load on first error (default) |
| `CONTINUE` | Skip bad rows, load everything else |
| `SKIP_FILE` | Skip the entire file if any error is found |
| `SKIP_FILE_<n>` | Skip file if error row count ≥ n |
| `SKIP_FILE_<n>%` | Skip file if error percentage ≥ n% |

In [ ]:
%%sql -r dataframe_6
-- VALIDATION_MODE: preview errors without writing any data to the table
COPY INTO pharmacy_claims
FROM @MY_STAGE/ingest_demo/csv_example/
FILES = ('pharmacy_claims_bad_records.csv')
FILE_FORMAT = (
  TYPE = CSV
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
)
VALIDATION_MODE = 'RETURN_ERRORS';

In [ ]:
%%sql -r dataframe_8
-- ON_ERROR = CONTINUE: skip bad rows and load everything else
COPY INTO pharmacy_claims
FROM @MY_STAGE/ingest_demo/csv_example/
FILES = ('pharmacy_claims_bad_records.csv')
FILE_FORMAT = (
  TYPE = CSV
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
)
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r dataframe_18
-- VALIDATE(): inspect which rows were skipped in the last COPY INTO job
SELECT *
FROM TABLE(VALIDATE(pharmacy_claims, JOB_ID => '_last'));

In [ ]:
%%sql -r dataframe_11
-- After dropping the incremental files, confirm rows were auto-ingested
SELECT COUNT(*) AS total_rows FROM pharmacy_claims;

## Snowpipe — Event-Driven Auto-Ingest & Schema Evolution

Snowpipe listens for Azure Event Notifications and automatically triggers a `COPY INTO` whenever a new file lands in the stage path. No manual scheduling needed.

### Snowpipe Setup
1. Configure Azure Queue + Azure Event Subscription pointing to that queue.
2. Create Snowflake Notification Integration. Accept on Azure side. Configure IAM.
3. Create the pipe with `AUTO_INGEST = TRUE` and `INTEGRATION = 'AZURE_SNOWPIPE_INTEGRATION'`
4. Drop files — Snowpipe ingests them automatically

---

### Schema Evolution

Drop `pharmacy_claims_add_refillnum.csv` into the stage. This file has **21 columns** — it adds `REFILL_NUMBER INTEGER` which doesn't exist in the current table.

Because `ENABLE_SCHEMA_EVOLUTION = TRUE` and the pipe uses `MATCH_BY_COLUMN_NAME`, Snowflake will automatically add the column when Snowpipe ingests the file.

In [ ]:
%%sql -r dataframe_9
-- Create the Snowpipe — any CSV that lands in the stage path is auto-ingested
CREATE OR REPLACE PIPE pipe_demo
  AUTO_INGEST = TRUE
  INTEGRATION = 'AZURE_SNOWPIPE_INTEGRATION'
AS
  COPY INTO pharmacy_claims
  FROM @my_stage/ingest_demo/csv_example/
  PATTERN = '.*\.csv$'
  FILE_FORMAT = (FORMAT_NAME = 'my_csv_file_format')
  MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

-- Get the notification_channel — this Azure Storage Queue URL goes into the Event Subscription

In [ ]:
%%sql -r dataframe_12
-- Step 1: Before upload — confirm table has 20 columns (no REFILL_NUMBER)
DESCRIBE TABLE pharmacy_claims;

### LOAD NEW FILE
file has 1k records + an additional column (schema evolution)

## XML Loading

Load `medical_claims.xml` into a raw `VARIANT` landing table, then flatten it into a typed structured table using `XMLGET`.

**Pattern:**
1. Load raw XML into a `VARIANT` column — one row per XML element
2. Use `XMLGET(src, 'TagName'):"$"::TYPE` to extract and cast each field

In [ ]:
%%sql -r dataframe_15
-- XML file format: STRIP_OUTER_ELEMENT removes the root wrapper tag
CREATE OR REPLACE FILE FORMAT MEDICAL_CLAIMS_XML_FF
  TYPE = XML
  STRIP_OUTER_ELEMENT = TRUE
  PRESERVE_SPACE = FALSE
  IGNORE_UTF8_ERRORS = FALSE;

-- Raw landing table: one VARIANT row per XML element + metadata columns
CREATE OR REPLACE TABLE MEDICAL_CLAIMS_RAW (
    src             VARIANT         NOT NULL,
    file_name       VARCHAR(500),
    load_timestamp  TIMESTAMP_NTZ   DEFAULT CURRENT_TIMESTAMP()
);

In [ ]:
%%sql -r dataframe_16
-- Load XML — METADATA$FILENAME captures which file each row came from
COPY INTO MEDICAL_CLAIMS_RAW (src, file_name)
    FROM (
      SELECT $1, METADATA$FILENAME
      FROM @my_stage/ingest_demo/xml_example/
    )
    FILES = ('medical_claims.xml')
    FILE_FORMAT = (FORMAT_NAME = MEDICAL_CLAIMS_XML_FF)
    ON_ERROR = ABORT_STATEMENT;


In [ ]:
%%sql -r dataframe_21
-- Inspect the raw VARIANT data
SELECT * FROM MEDICAL_CLAIMS_RAW;

In [ ]:
%%sql -r dataframe_17
-- Flatten XML into a typed structured table using XMLGET
-- Filter out non-claim rows (e.g., `<BatchInfo>`)
-- XMLGET(src, 'TagName'):"$"::TYPE extracts the text content of each XML element
CREATE OR REPLACE TABLE MEDICAL_CLAIMS_STRUCTURED AS (
SELECT
    XMLGET(src, 'ClaimID')             :"$"::VARCHAR(20)    AS claim_id,
    XMLGET(src, 'MemberID')            :"$"::VARCHAR(20)    AS member_id,
    XMLGET(src, 'DateOfBirth')         :"$"::DATE           AS date_of_birth,
    XMLGET(src, 'Gender')              :"$"::VARCHAR(1)     AS gender,
    XMLGET(src, 'PlanType')            :"$"::VARCHAR(10)    AS plan_type,
    XMLGET(src, 'ServiceDate')         :"$"::DATE           AS service_date,
    XMLGET(src, 'ProviderNPI')         :"$"::VARCHAR(10)    AS provider_npi,
    XMLGET(src, 'ProviderName')        :"$"::VARCHAR(100)   AS provider_name,
    XMLGET(src, 'ProviderSpecialty')   :"$"::VARCHAR(50)    AS provider_specialty,
    XMLGET(src, 'PlaceOfService')      :"$"::VARCHAR(2)     AS place_of_service,
    XMLGET(src, 'DiagnosisCode1')      :"$"::VARCHAR(10)    AS diagnosis_code_1,
    XMLGET(src, 'DiagnosisCode2')      :"$"::VARCHAR(10)    AS diagnosis_code_2,
    XMLGET(src, 'ProcedureCode')       :"$"::VARCHAR(5)     AS procedure_code,
    XMLGET(src, 'ProcedureDescription'):"$"::VARCHAR(100)   AS procedure_description,
    XMLGET(src, 'Units')               :"$"::INTEGER        AS units,
    XMLGET(src, 'BilledAmount')        :"$"::NUMBER(10,2)   AS billed_amount,
    XMLGET(src, 'AllowedAmount')       :"$"::NUMBER(10,2)   AS allowed_amount,
    XMLGET(src, 'PlanPaidAmount')      :"$"::NUMBER(10,2)   AS plan_paid_amount,
    XMLGET(src, 'MemberResponsibility'):"$"::NUMBER(10,2)   AS member_responsibility,
    XMLGET(src, 'ClaimStatus')         :"$"::VARCHAR(10)    AS claim_status,
    file_name,
    load_timestamp
FROM MEDICAL_CLAIMS_RAW
WHERE XMLGET(src, 'ClaimID') IS NOT NULL   -- exclude the <BatchInfo> wrapper row
);

SELECT * FROM MEDICAL_CLAIMS_STRUCTURED;

## Verify Snowpipe Loaded file + Evolved Schema

In [ ]:
%%sql -r dataframe_19
SHOW PIPES;

SELECT "name", "notification_channel" AS queue
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

In [ ]:
%%sql -r dataframe_10
-- Check pipe health and pending file queue
SELECT SYSTEM$PIPE_STATUS('pipe_demo');

In [ ]:
%%sql -r dataframe_20
-- Step 1: After Snowpipe ingests — REFILL_NUMBER appears as a new INTEGER column
DESCRIBE TABLE pharmacy_claims;

In [ ]:
%%sql -r dataframe_13
-- Step 4: Check the data 
SELECT
    CLAIM_ID,
    DRUG_NAME,
    DAYS_SUPPLY,
    REFILL_NUMBER --New Column via Schema evolution.
FROM pharmacy_claims
WHERE REFILL_NUMBER IS NOT NULL 
LIMIT 20;